# Project Summary (Data + Features)

This notebook summarizes what has been built so far:
- Raw weather data coverage
- Daily energy aggregation
- Wastage target creation
- Feature engineering output
- Final dataset readiness for ML

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

print('✓ Libraries loaded')

✓ Libraries loaded


## 1. Daily Energy Dataset

In [2]:
daily_path = Path('../data/processed/daily_energy.csv')
daily = pd.read_csv(daily_path)
daily['date'] = pd.to_datetime(daily['date'])

print(f'Path: {daily_path}')
print(f'Shape: {daily.shape}')
print(f'Date range: {daily["date"].min()} to {daily["date"].max()}')
print(f'Households: {daily["household"].nunique()}')
print('\nHead:')
print(daily.head())

Path: ..\data\processed\daily_energy.csv
Shape: (5631, 4)
Date range: 2014-12-11 00:00:00 to 2019-05-01 00:00:00
Households: 4

Head:
        date  daily_pv_kwh  daily_load_kwh     household
0 2015-05-21      0.010838        0.018602  residential1
1 2015-05-22      0.555957        0.214720  residential1
2 2015-05-23      1.382767        0.409031  residential1
3 2015-05-24      2.177085        0.574465  residential1
4 2015-05-25      2.962496        0.684174  residential1


## 2. Weather Data (Combined)

In [3]:
weather_path = Path('../data/processed/weather_all_areas.csv')
weather = pd.read_csv(weather_path)
weather['date'] = pd.to_datetime(weather['date'])

print(f'Path: {weather_path}')
print(f'Shape: {weather.shape}')
print(f'Date range: {weather["date"].min()} to {weather["date"].max()}')
print(f'Areas: {weather["area"].unique()}')
print('\nHead:')
print(weather.head())

print('\nWeather summary (by area):')
print(weather.groupby('area')[['irradiance', 'temp']].describe().round(2))


Path: ..\data\processed\weather_all_areas.csv
Shape: (6570, 4)
Date range: 2014-01-02 00:00:00 to 2019-12-31 00:00:00
Areas: ['Colombo' 'Galle' 'Matara']

Head:
        date  irradiance   temp     area
0 2014-01-02      5.5836  24.98  Colombo
1 2014-01-03      5.1614  24.91  Colombo
2 2014-01-04      5.2354  24.89  Colombo
3 2014-01-05      5.9659  24.42  Colombo
4 2014-01-06      5.4946  24.77  Colombo

Weather summary (by area):
        irradiance                                              temp         \
             count  mean   std   min   25%   50%   75%   max   count   mean   
area                                                                          
Colombo     2190.0  5.63  1.22  0.54  5.05  5.93  6.47  7.65  2190.0  26.52   
Galle       2190.0  5.07  1.07  0.81  4.49  5.20  5.79  7.62  2190.0  27.49   
Matara      2190.0  5.63  1.29  0.59  5.06  5.98  6.52  7.61  2190.0  27.09   

                                                  
          std    min    25%    50%    7

## 3. Feature-Engineered Dataset

In [4]:
features_path = Path('../data/processed/features_engineered1.csv')
features = pd.read_csv(features_path)
features['date'] = pd.to_datetime(features['date'])

print(f'Path: {features_path}')
print(f'Shape: {features.shape}')
print(f'Date range: {features["date"].min()} to {features["date"].max()}')
print(f'Columns: {len(features.columns)}')
print('\nHead:')
print(features.head())


Path: ..\data\processed\features_engineered1.csv
Shape: (5631, 44)
Date range: 2014-12-11 00:00:00 to 2019-05-01 00:00:00
Columns: 44

Head:
        date  daily_pv_kwh  daily_load_kwh     household  net_export  \
0 2015-05-21      0.010838        0.018602  residential1   -0.007764   
1 2015-05-22      0.555957        0.214720  residential1    0.341237   
2 2015-05-23      1.382767        0.409031  residential1    0.973736   
3 2015-05-24      2.177085        0.574465  residential1    1.602620   
4 2015-05-25      2.962496        0.684174  residential1    2.278322   

   wasted_energy_kwh  waste_flag  wasted_energy_log1p  year  month  day  \
0                0.0           0                  0.0  2015      5   21   
1                0.0           0                  0.0  2015      5   22   
2                0.0           0                  0.0  2015      5   23   
3                0.0           0                  0.0  2015      5   24   
4                0.0           0                  0

## 4. Target (Wastage) Summary

In [5]:
target = 'wasted_energy_kwh'
print(features[target].describe())
print(f'Days with wastage > 0: {(features[target] > 0).sum()}')
print(f'Total wasted energy: {features[target].sum():.2f} kWh')

count    5631.000000
mean       85.272133
std       131.265073
min         0.000000
25%         0.000000
50%         0.000000
75%       134.695391
max       507.759398
Name: wasted_energy_kwh, dtype: float64
Days with wastage > 0: 2583
Total wasted energy: 480167.38 kWh


## 5. Feature Inventory

In [6]:
# Show feature groups
cols = features.columns.tolist()
time_features = [c for c in cols if c in ['year','month','day','quarter','day_of_week','day_of_year','week_of_year','is_weekend','season']]
lag_features = [c for c in cols if 'lag_' in c]
rolling_features = [c for c in cols if 'rolling_' in c]
weather_features = [c for c in cols if c in ['irradiance','temp']]
id_features = [c for c in cols if c in ['date','household','district']]
energy_features = [c for c in cols if c in ['daily_pv_kwh','daily_load_kwh','net_export','export_limit_kwh']]

print(f'Time features ({len(time_features)}): {time_features}')
print(f'Lag features ({len(lag_features)}): {lag_features[:10]} ...')
print(f'Rolling features ({len(rolling_features)}): {rolling_features[:10]} ...')
print(f'Weather features ({len(weather_features)}): {weather_features}')
print(f'Energy features ({len(energy_features)}): {energy_features}')
print(f'ID features ({len(id_features)}): {id_features}')

Time features (9): ['year', 'month', 'day', 'quarter', 'day_of_week', 'day_of_year', 'week_of_year', 'is_weekend', 'season']
Lag features (12): ['pv_lag_1d', 'load_lag_1d', 'net_export_lag_1d', 'wastage_lag_1d', 'pv_lag_3d', 'load_lag_3d', 'net_export_lag_3d', 'wastage_lag_3d', 'pv_lag_7d', 'load_lag_7d'] ...
Rolling features (12): ['pv_rolling_mean_7d', 'load_rolling_mean_7d', 'pv_rolling_std_7d', 'wastage_rolling_mean_7d', 'pv_rolling_mean_14d', 'load_rolling_mean_14d', 'pv_rolling_std_14d', 'wastage_rolling_mean_14d', 'pv_rolling_mean_30d', 'load_rolling_mean_30d'] ...
Weather features (2): ['irradiance', 'temp']
Energy features (3): ['daily_pv_kwh', 'daily_load_kwh', 'net_export']
ID features (3): ['date', 'household', 'district']


## 6. Data Quality Checks

In [7]:
missing = features.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print('Missing values (if any):')
print(missing if len(missing) > 0 else 'None')

print('\nDuplicate rows:', features.duplicated().sum())

print('\nHouseholds count:')
print(features['household'].value_counts())


Missing values (if any):
None

Duplicate rows: 0

Households count:
household
residential3    1603
residential1    1442
residential4    1300
residential6    1286
Name: count, dtype: int64


## 7. Ready for Model Training

If everything looks correct above, proceed to the model training notebook: 03_model_training.ipynb

## 8. Regression-Ready Dataset (Leakage-Safe)

This step creates a modeling-ready dataset from `features_engineered1.csv` for direct wastage amount regression.

- Target: `wasted_energy_kwh`
- Drop from features: `date`, `wasted_energy_kwh`, `waste_flag`, `wasted_energy_log1p`, `net_export`
- Output file: `../data/processed/features_regression_ready.csv`

In [ ]:
from pathlib import Path
import pandas as pd

reg_in_path = Path('../data/processed/features_engineered1.csv')
reg_out_path = Path('../data/processed/features_regression_ready.csv')

reg_df = pd.read_csv(reg_in_path)

drop_from_X = [
    'date',
    'wasted_energy_kwh',
    'waste_flag',
    'wasted_energy_log1p',
    'net_export'
]

reg_X = reg_df.drop(columns=[c for c in drop_from_X if c in reg_df.columns]).copy()
reg_y = reg_df['wasted_energy_kwh'].copy()

reg_ready = reg_X.copy()
reg_ready.insert(0, 'wasted_energy_kwh', reg_y)

reg_ready.to_csv(reg_out_path, index=False)

print(f'Input file: {reg_in_path}')
print(f'Output file: {reg_out_path}')
print(f'Input shape: {reg_df.shape}')
print(f'Output shape: {reg_ready.shape}')
print(f'Dropped columns: {[c for c in drop_from_X if c in reg_df.columns]}')
print(f'Has target column: {"wasted_energy_kwh" in reg_ready.columns}')
print(f'Has leakage column net_export: {"net_export" in reg_ready.columns}')

NameError: name 'Path' is not defined